In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q shap imbalanced-learn streamlit

import os
os.chdir('/content/drive/MyDrive/network-intrusion-ml')
print("Working folder:", os.getcwd())
print(os.listdir())

Mounted at /content/drive
Working folder: /content/drive/MyDrive/network-intrusion-ml
['data', 'notebooks', 'src', 'models', 'results']


In [2]:
%%writefile src/utils.py
"""
utils.py - shared helper functions. Every notebook imports this so that
all charts, tables, models and experiment logs are saved automatically
into Google Drive.
"""
from pathlib import Path
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

# Project root inside Google Drive
ROOT = Path("/content/drive/MyDrive/network-intrusion-ml")

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIG_DIR = ROOT / "results" / "figures"
TABLE_DIR = ROOT / "results" / "tables"
METRIC_DIR = ROOT / "results" / "metrics"
MODEL_DIR = ROOT / "models"
LOG_FILE = ROOT / "results" / "experiment_log.csv"

# Make sure every folder exists
for d in [RAW_DIR, PROCESSED_DIR, FIG_DIR, TABLE_DIR, METRIC_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42  # fixed seed so results are repeatable


def save_fig(name, dpi=150):
    """Save the current matplotlib figure to results/figures/<name>.png"""
    path = FIG_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"[saved figure] {path}")


def save_table(df, name, index=True):
    """Save a DataFrame to results/tables/<name>.csv"""
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print(f"[saved table] {path}")


def log_experiment(experiment_id, task, model, features, preprocessing,
                   hyperparameters, split, metrics, notes=""):
    """Append one row to results/experiment_log.csv (the project document requires this)."""
    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_id": experiment_id,
        "task": task,
        "model": model,
        "features": features,
        "preprocessing": preprocessing,
        "hyperparameters": str(hyperparameters),
        "split": split,
        "metrics": str(metrics),
        "notes": notes,
    }
    df = pd.DataFrame([row])
    df.to_csv(LOG_FILE, mode="a", header=not LOG_FILE.exists(), index=False)
    print(f"[logged experiment] {experiment_id}")

Writing src/utils.py


In [3]:
import sys
sys.path.append('/content/drive/MyDrive/network-intrusion-ml')
from src.utils import RAW_DIR

files = sorted(RAW_DIR.glob("*.csv"))
print(len(files), "CSV files found:")
for f in files:
    print(" -", f.name, round(f.stat().st_size / 1e6), "MB")

8 CSV files found:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv 77 MB
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv 77 MB
 - Friday-WorkingHours-Morning.pcap_ISCX.csv 58 MB
 - Monday-WorkingHours.pcap_ISCX.csv 177 MB
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv 83 MB
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv 52 MB
 - Tuesday-WorkingHours.pcap_ISCX.csv 135 MB
 - Wednesday-workingHours.pcap_ISCX.csv 225 MB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content/drive/MyDrive/network-intrusion-ml')
sys.path.append('/content/drive/MyDrive/network-intrusion-ml')

import numpy as np
import pandas as pd

from src.utils import RAW_DIR, PROCESSED_DIR, save_table

print("Ready. Working folder:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready. Working folder: /content/drive/MyDrive/network-intrusion-ml


In [5]:
frames = []
for f in sorted(RAW_DIR.glob("*.csv")):
    part = pd.read_csv(f, encoding="latin1", low_memory=False)
    part.columns = part.columns.str.strip()          # column names have stray spaces
    num_cols = [c for c in part.columns if c != "Label"]
    part[num_cols] = part[num_cols].astype("float32") # halves memory use
    frames.append(part)
    print(f"{f.name}: {part.shape[0]:,} rows, {part.shape[1]} columns")

df = pd.concat(frames, ignore_index=True)
del frames

# Some label names contain a broken symbol (Web Attack � Brute Force). Fix it.
df["Label"] = df["Label"].str.replace(r"[^\x00-\x7F]+", "-", regex=True).str.strip()

print("\nMerged table:", f"{df.shape[0]:,}", "rows x", df.shape[1], "columns")

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows, 79 columns
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows, 79 columns
Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows, 79 columns
Monday-WorkingHours.pcap_ISCX.csv: 529,918 rows, 79 columns
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 rows, 79 columns
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 rows, 79 columns
Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 rows, 79 columns
Wednesday-workingHours.pcap_ISCX.csv: 692,703 rows, 79 columns

Merged table: 2,830,743 rows x 79 columns


In [6]:
feature_cols = [c for c in df.columns if c != "Label"]

print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])
print("Target column: Label")
print("\nData types:")
print(df.dtypes.value_counts())

# One row per column: type, missing, infinite, unique values
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isnull().sum(),
    "infinite": np.isinf(df[feature_cols]).sum().reindex(df.columns).fillna(0).astype(int),
    "unique_values": df.nunique(),
})
print("\nColumns with missing or infinite values:")
print(profile[(profile["missing"] > 0) | (profile["infinite"] > 0)])

dups = df.duplicated().sum()
print("\nDuplicate rows:", f"{dups:,}")

class_dist = df["Label"].value_counts().to_frame("count")
class_dist["percent"] = (class_dist["count"] / len(df) * 100).round(3)
print("\nClass distribution:")
print(class_dist)

# Save everything
save_table(profile, "01_feature_profile")
save_table(class_dist, "01_class_distribution_raw")

Rows: 2,830,743
Columns: 79
Target column: Label

Data types:
float32    78
object      1
Name: count, dtype: int64

Columns with missing or infinite values:
                  dtype  missing  infinite  unique_values
Flow Bytes/s    float32     1358      1509        1527320
Flow Packets/s  float32        0      2867        1165221

Duplicate rows: 331,200

Class distribution:
                              count  percent
Label                                       
BENIGN                      2273097   80.300
DoS Hulk                     231073    8.163
PortScan                     158930    5.614
DDoS                         128027    4.523
DoS GoldenEye                 10293    0.364
FTP-Patator                    7938    0.280
SSH-Patator                    5897    0.208
DoS slowloris                  5796    0.205
DoS Slowhttptest               5499    0.194
Bot                            1966    0.069
Web Attack - Brute Force       1507    0.053
Web Attack - XSS                652  

In [8]:
decisions = []
feature_cols = [c for c in df.columns if c != "Label"]
start_rows, start_cols = len(df), df.shape[1]

# 1. Duplicate column: the dataset has "Fwd Header Length" twice
if "Fwd Header Length.1" in df.columns:
    if df["Fwd Header Length"].equals(df["Fwd Header Length.1"]):
        df = df.drop(columns=["Fwd Header Length.1"])
        decisions.append(["Dropped duplicate column", "Fwd Header Length.1 was identical to Fwd Header Length", 1])
    else:
        decisions.append(["Kept 'Fwd Header Length.1'", "not identical to the original, so not dropped", 0])
feature_cols = [c for c in df.columns if c != "Label"]

# 2. Infinite values -> NaN, then look at which labels they belong to
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
bad_rows = df[feature_cols].isnull().any(axis=1)
print("Rows with NaN/inf:", f"{bad_rows.sum():,}")
print("Their labels:")
print(df.loc[bad_rows, "Label"].value_counts())

# Decision: drop them (a tiny share of the data; filling them in would need
# statistics that must be computed on the training set only)
decisions.append(["Dropped rows with NaN/inf", "tiny share of data; avoids guessing values", int(bad_rows.sum())])
df = df.loc[~bad_rows].copy()

# 3. Exact duplicate rows
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates()
decisions.append(["Dropped duplicate rows", "prevents identical rows landing in both train and test", n_dup])

# 4. Constant columns (same value in every row = no information)
const_cols = [c for c in feature_cols if df[c].nunique() <= 1]
df = df.drop(columns=const_cols)
decisions.append(["Dropped constant columns", ", ".join(const_cols) if const_cols else "none found", len(const_cols)])
feature_cols = [c for c in df.columns if c != "Label"]

# 5. Negative values: only REPORT them. Some (like Init_Win_bytes = -1) are normal.
neg_counts = (df[feature_cols] < 0).sum()
neg_counts = neg_counts[neg_counts > 0]
print("\nColumns with negative values (kept, not removed):")
print(neg_counts)
decisions.append(["Kept negative values", "some are valid codes; outliers/extremes may be real attack behaviour", int(neg_counts.sum())])

# 6. Group the 15 original labels into the 10 categories from your project document
def group_label(label):
    if label.startswith("DoS"):
        return "DoS"
    if label.startswith("Web Attack"):
        return "Web Attack"
    return label

df["Label_Group"] = df["Label"].map(group_label)
decisions.append(["Added Label_Group column", "DoS variants -> DoS, Web Attack variants -> Web Attack", 0])

df = df.reset_index(drop=True)

# Summary
print("\nRows:   ", f"{start_rows:,}", "->", f"{len(df):,}")
print("Columns:", start_cols, "->", df.shape[1])
print("\nOriginal labels:")
print(df["Label"].value_counts())
print("\nGrouped labels (Label_Group):")
print(df["Label_Group"].value_counts())

# Save everything
decision_log = pd.DataFrame(decisions, columns=["decision", "reason", "count_affected"])
save_table(decision_log, "01_cleaning_decisions", index=False)
save_table(df["Label"].value_counts().to_frame("count"), "01_class_distribution_clean")
save_table(df["Label_Group"].value_counts().to_frame("count"), "01_label_group_distribution_clean")

df.to_parquet(PROCESSED_DIR / "cicids2017_clean.parquet", index=False)
print("\n[saved cleaned dataset]", PROCESSED_DIR / "cicids2017_clean.parquet")

Rows with NaN/inf: 0
Their labels:
Series([], Name: count, dtype: int64)

Columns with negative values (kept, not removed):
Flow Duration                  107
Flow Bytes/s                    78
Flow Packets/s                 107
Flow IAT Mean                  107
Flow IAT Max                   107
Flow IAT Min                  2880
Fwd IAT Min                     17
Fwd Header Length               35
Bwd Header Length               22
Init_Win_bytes_forward      892949
Init_Win_bytes_backward    1194110
min_seg_size_forward            35
dtype: int64

Rows:    2,827,876 -> 2,497,980
Columns: 78 -> 71

Original labels:
Label
BENIGN                        2072254
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10282
FTP-Patator                      5931
DoS slowloris                    5374
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                       

In [9]:
import numpy as np
import pandas as pd
from src.utils import RAW_DIR, PROCESSED_DIR, save_table

clean = pd.read_parquet(PROCESSED_DIR / "cicids2017_clean.parquet")

# Find which columns were dropped by comparing with the raw file header
first_csv = sorted(RAW_DIR.glob("*.csv"))[0]
raw_cols = pd.read_csv(first_csv, encoding="latin1", nrows=1).columns.str.strip().tolist()
dropped = [c for c in raw_cols if c not in clean.columns]
dup_col = [c for c in dropped if c.endswith(".1")]
const_cols = [c for c in dropped if not c.endswith(".1")]

rows_raw = 2_830_743
bad_rows = 2_867                      # 1,358 NaN + 1,509 inf rows from Cell 3
dups_removed = rows_raw - bad_rows - len(clean)

decisions = [
    ["Dropped duplicate column", "identical copy of Fwd Header Length: " + ", ".join(dup_col), len(dup_col)],
    ["Dropped rows with NaN/inf", "all in Flow Bytes/s and Flow Packets/s (division by zero); tiny share of data", bad_rows],
    ["Dropped duplicate rows", "prevents identical rows in both train and test", dups_removed],
    ["Dropped constant columns", ", ".join(const_cols), len(const_cols)],
    ["Kept negative values", "Init_Win_bytes = -1 is a valid code; ~100 negative time values are too few to matter", 0],
    ["Added Label_Group column", "DoS variants -> DoS, Web Attack variants -> Web Attack", 0],
]
log = pd.DataFrame(decisions, columns=["decision", "reason", "count_affected"])
save_table(log, "01_cleaning_decisions", index=False)
print(log.to_string(index=False))

# Safety check: the cleaned data must have no NaN or inf
feat = clean.drop(columns=["Label", "Label_Group"])
print("\nFinal shape:", clean.shape)
print("NaN left:", int(feat.isnull().sum().sum()), "| inf left:", int(np.isinf(feat).sum().sum()))

[saved table] /content/drive/MyDrive/network-intrusion-ml/results/tables/01_cleaning_decisions.csv
                 decision                                                                                                                                                 reason  count_affected
 Dropped duplicate column                                                                                               identical copy of Fwd Header Length: Fwd Header Length.1               1
Dropped rows with NaN/inf                                                                          all in Flow Bytes/s and Flow Packets/s (division by zero); tiny share of data            2867
   Dropped duplicate rows                                                                                                         prevents identical rows in both train and test          329896
 Dropped constant columns Bwd PSH Flags, Bwd URG Flags, Fwd Avg Bytes/Bulk, Fwd Avg Packets/Bulk, Fwd Avg Bulk Rate, Bwd Avg Byte

In [10]:
import numpy as np
import pandas as pd
from src.utils import RAW_DIR

parts = []
for f in sorted(RAW_DIR.glob("*.csv")):
    p = pd.read_csv(f, encoding="latin1",
                    usecols=lambda c: c.strip() in ("Flow Packets/s", "Label"))
    p.columns = p.columns.str.strip()
    bad = ~np.isfinite(p["Flow Packets/s"].astype("float64"))
    labels = p.loc[bad, "Label"].str.replace(r"[^\x00-\x7F]+", "-", regex=True).str.strip()
    parts.append(labels)

dropped_labels = pd.concat(parts).value_counts()
print("Labels of the 2,867 dropped rows:")
print(dropped_labels)

Labels of the 2,867 dropped rows:
Label
BENIGN         1777
DoS Hulk        949
PortScan        126
Bot              10
FTP-Patator       3
DDoS              2
Name: count, dtype: int64
